In [24]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

In [52]:
%load_ext sql

In [36]:
spark.stop()

In [37]:
spark = (SparkSession
    .builder
    .config("spark.jars", r"../misc/postgresql-42.7.9.jar")
    .appName("hw01")
    .getOrCreate()
)

In [39]:
trip_data = spark.read.parquet("../data/green_tripdata_2025-11.parquet")
zone_lookup = spark.read.csv("../data/taxi_zone_lookup.csv", header=True)

In [40]:
trip_data.show(5)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|cbd_congestion_fee|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+
|       2| 2025-11-01 00:34:48|  2025-11-01 00:41:39|                 N|         1|          74|          

In [41]:
trip_data.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- trip_type: long (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [48]:
trip_data.createOrReplaceTempView("trip_data")

In [49]:
zone_lookup.createOrReplaceTempView("zone_lookup")

In [59]:
spark.sql("""
select count(*) as total_trips
from trip_data
where true
    and (lpep_pickup_datetime >= '2025-11-01' and lpep_pickup_datetime < '2025-12-01')
    and trip_distance <= 1
""").show()

+-----------+
|total_trips|
+-----------+
|       8007|
+-----------+



In [64]:
spark.sql("""
select lpep_pickup_datetime, trip_distance
from trip_data
where true
    and trip_distance <= 100
ORDER BY trip_distance DESC
LIMIT 1
""").show()

+--------------------+-------------+
|lpep_pickup_datetime|trip_distance|
+--------------------+-------------+
| 2025-11-14 15:36:27|        88.03|
+--------------------+-------------+



In [70]:
zone_lookup.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [74]:
spark.sql("""
select zone, sum(total_amount) total_amount
from trip_data
LEFT JOIN zone_lookup
    ON trip_data.PULocationID = zone_lookup.LocationID
where true
    and (lpep_pickup_datetime >= '2025-11-18' and lpep_pickup_datetime < '2025-11-19')
GROUP BY ALL
ORDER BY total_amount DESC
""").show()

+--------------------+------------------+
|                zone|      total_amount|
+--------------------+------------------+
|   East Harlem North| 9281.919999999996|
|   East Harlem South| 6696.130000000003|
|        Central Park|2378.7899999999995|
|Washington Height...|           2139.05|
| Morningside Heights|2100.5900000000006|
|             Jamaica|1998.1100000000001|
|         Fort Greene|1780.4099999999999|
|Downtown Brooklyn...|           1499.02|
|        Forest Hills|1423.7500000000005|
|            Elmhurst|1251.8199999999997|
|      Central Harlem|           1136.02|
|            Flushing|            948.87|
|          Park Slope| 882.7299999999999|
|     Jackson Heights| 653.1899999999999|
|Central Harlem North| 617.3799999999999|
|    Brooklyn Heights|            521.92|
|Upper East Side N...|504.84999999999997|
|         Kew Gardens|444.73999999999995|
|Flushing Meadows-...|            432.64|
|            Steinway|            387.44|
+--------------------+------------